In [0]:
import pandas as pd 
from matplotlib import pyplot as plt 
import seaborn as sns 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
# from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import FunctionTransformer
from mlflow.models import infer_signature
import mlflow
import mlflow.sklearn
import numpy as np

In [0]:
from etl import load_dataSet
TRAINING_PATH = "/Volumes/workspace/default/data_customers/training_parquet"
TESTING_PATH = "/Volumes/workspace/default/data_customers/testing_parquet"
df_train = load_dataSet(spark, TRAINING_PATH)
df_test = load_dataSet(spark, TESTING_PATH)

In [0]:
df_train = df_train.drop("CustomerID")
df_test = df_test.drop("CustomerID")

In [0]:
df_train.printSchema()

In [0]:
display(df_train)

In [0]:
df_test.printSchema()

In [0]:
display(df_test)

In [0]:
from pyspark.sql.functions import col, sum, when

missing_df_train= df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
])

display(missing_df_train)

In [0]:
df_train = df_train.dropna()
df_test = df_test.dropna()


display(df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
]))


In [0]:
df_train = df_train.dropDuplicates()
df_test = df_test.dropDuplicates()

In [0]:
for c in df_train.columns:
    print(c, df_train.select(c).distinct().count())

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

churn_pd = df_train.select("Churn").toPandas()

plt.figure(figsize=(6,4))
ax = sns.countplot(x='Churn', data=churn_pd)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width()/2., p.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [0]:
num_cols = [
    f.name for f in df_train.schema.fields
    if f.dataType.simpleString() in ["int", "double"]
    and f.name != "Churn"
]

print("Numeric columns:", num_cols)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in num_cols:
    
    # convert cột Spark -> Pandas
    data = df_train.select(col).toPandas()[col]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(data, bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"Histogram of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.3)

    # KDE
    sns.kdeplot(data, fill=True, ax=axes[1], color='orange')
    axes[1].set_title(f"KDE of {col}")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Density")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Nếu data lớn thì sample trước
sample_df = df_train.sample(fraction=0.2, seed=42)

for col in num_cols:
    
    # Spark -> Pandas
    data = sample_df.select(col).toPandas()[col]

    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    # Boxplot
    axes[0].boxplot(
        data,
        vert=False,
        patch_artist=True,
        boxprops=dict(facecolor='skyblue', alpha=0.7),
        medianprops=dict(color='red')
    )
    axes[0].set_title(f"Boxplot of {col}")
    axes[0].set_xlabel(col)
    axes[0].grid(axis='x', alpha=0.3)

    # Violin Plot
    sns.violinplot(
        x=data,
        ax=axes[1],
        inner='quartile',
        color='orange'
    )
    axes[1].set_title(f"Violin Plot of {col}")
    axes[1].set_xlabel(col)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

In [0]:
from pyspark.sql.functions import avg

churn_rate = (
    df_train
    .groupBy("Support Calls")
    .agg(avg("Churn").alias("churn_rate"))
    .orderBy("Support Calls")
)

churn_pd = churn_rate.toPandas()

plt.figure(figsize=(8,5))
plt.bar(churn_pd['Support Calls'], churn_pd['churn_rate'])

plt.xlabel("Number of Support Calls")
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Support Calls")
plt.ylim(0,1)

plt.show()

In [0]:
cols = num_cols + ["Churn"]
pdf = df_train.select(cols).toPandas()

corr = pdf.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix including Churn")
plt.show()

### Check Categorical Columns

In [0]:
categorical_cols = [f.name for f in df_train.schema.fields 
                    if f.dataType.simpleString() == 'string']
display(categorical_cols)

In [0]:
for col in categorical_cols:
    print(f"\n===== {col} =====")
    display(df_train.groupBy(col).count().orderBy("count", ascending=False))

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert toàn bộ categorical columns sang pandas một lần
pdf = df_train.select(categorical_cols).toPandas()

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    
    sns.countplot(
        x=pdf[col],
        order=pdf[col].value_counts().index
    )
    
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()